# AI for Data Analysts with ChatGPT  
## Section 3: AI-Assisted pandas Programming

**Instructor Edition**  
Course: UCI CDE Python for Data Analysis  
Case: BrightMart Retail Analytics

This section teaches students how to use AI to draft pandas code, then review, improve, and verify that code.

## Section 3 Learning Objectives

By the end of this section, students should be able to:

1. Write effective prompts for pandas programming tasks.
2. Generate pandas code for common business analytics questions.
3. Review AI-generated code for correctness and readability.
4. Verify grouped totals against source data.
5. Translate pandas output into business interpretation.

## Instructor Notes

This section should be taught as an **AI-assisted workflow**, not a copy-and-paste coding exercise.

Emphasize this cycle:

1. Define the business question.
2. Ask AI for code with context.
3. Run the code.
4. Verify the output.
5. Improve readability.
6. Explain the result in business language.

**Teaching tip:** Ask students to compare their own code to AI-generated code. The goal is not to prove AI is better. The goal is to learn how to review code critically.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis'
DATA_DIR = f'{BASE_DIR}/05_Datasets'
ASSIGNMENTS_DIR = f'{BASE_DIR}/04_Assignments'
OUTPUTS_DIR = f'{BASE_DIR}/04_Assignments/Outputs'

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis/07_Capstone_Project/UCI_CDE_Capstone_Package1B/BrightMart_Retail_Dataset.csv")
df.head()

,OrderID,OrderDate,Region,Category,CustomerSegment,Sales,Discount,Profit,Quantity
0,100000,2025-06-15,West,Technology,Consumer,"4,740.37",0.15,710.59,2
1,100001,2025-08-11,South,Office Supplies,Consumer,"2,102.50",0.05,-121.23,10
2,100002,2025-10-27,East,Office Supplies,Consumer,"1,995.47",0.05,-152.93,2
3,100003,2025-10-25,West,Furniture,Corporate,"2,864.60",0.05,181.80,10
4,100004,2025-08-21,East,Furniture,Corporate,"1,820.68",0.05,445.91,10


In [6]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis'
DATA_DIR = f'{BASE_DIR}/05_Datasets'
ASSIGNMENTS_DIR = f'{BASE_DIR}/04_Assignments'
OUTPUTS_DIR = f'{BASE_DIR}/04_Assignments/Outputs'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Prompt Pattern for pandas Tasks

A strong pandas prompt should include:

- the business goal
- the dataframe name
- relevant column names
- expected output format
- verification requirements

### Reusable Prompt Template

> I am using pandas with a dataframe named `df`.  
> The relevant columns are: [list columns].  
> My business question is: [question].  
> Please write readable pandas code using method chaining where appropriate.  
> Also include one verification check to confirm the result is correct.

## Exercise 1 — Calculate Monthly Revenue

### Business Question

How much revenue did BrightMart generate each month?

### Prompt to ChatGPT

> I am using pandas with a dataframe named `df`. The columns include `OrderDate` and `Sales`.  
> Write pandas code to convert `OrderDate` to datetime, calculate monthly revenue, sort the results by month, and verify that monthly revenue totals equal the original sales total.

In [7]:
# Instructor solution
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
df["OrderMonth"] = df["OrderDate"].dt.to_period("M").astype(str)

monthly_revenue = (
    df.groupby("OrderMonth", as_index=False)
      .agg(MonthlyRevenue=("Sales", "sum"))
      .sort_values("OrderMonth")
)

monthly_revenue.head()

,OrderMonth,MonthlyRevenue
0,2025-01,"833,921.57"
1,2025-02,"978,054.78"
2,2025-03,"737,339.48"
3,2025-04,"799,236.28"
4,2025-05,"700,164.81"


In [8]:
# Expected verification output: these two totals should match or be extremely close
raw_total_sales = df["Sales"].sum()
monthly_total_sales = monthly_revenue["MonthlyRevenue"].sum()

print("Raw total sales:", round(raw_total_sales, 2))
print("Monthly grouped total sales:", round(monthly_total_sales, 2))
print("Difference:", round(raw_total_sales - monthly_total_sales, 2))

Raw total sales: 10063685.83
Monthly grouped total sales: 10063685.83
Difference: -0.0


### Teaching Checkpoint

Students should identify that date conversion is required before monthly grouping. If AI skips date conversion, the code may still run but produce poor time-grouping logic.

## Exercise 2 — Find Top Performing Categories

### Business Question

Which product categories generate the most sales and profit?

### Prompt to ChatGPT

> I am analyzing a retail dataframe named `df` with columns `Category`, `Sales`, `Profit`, and `OrderID`.  
> Write pandas code to summarize total sales, total profit, order count, and profit margin by category. Sort by profit from highest to lowest.

In [9]:
category_summary = (
    df.groupby("Category")
      .agg(
          Orders=("OrderID", "count"),
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
      .assign(ProfitMargin=lambda x: x["Profit"] / x["Sales"])
      .sort_values("Profit", ascending=False)
)

category_summary

,Orders,Sales,Profit,ProfitMargin
Category,,,,
Office Supplies,1185,"3,534,425.29","250,737.92",0.07
Furniture,1164,"3,262,111.93","249,788.64",0.08
Technology,1186,"3,267,148.61","237,562.42",0.07


In [10]:
# Verification: category totals should match the raw totals
print("Sales check:", round(category_summary["Sales"].sum() - df["Sales"].sum(), 2))
print("Profit check:", round(category_summary["Profit"].sum() - df["Profit"].sum(), 2))

Sales check: 0.0
Profit check: 0.0


### Expected Output

A category-level table with:

- Orders
- Sales
- Profit
- ProfitMargin

The verification differences should be `0.0` or close to zero.

## Exercise 3 — Region Performance Summary

### Business Question

Which regions are most profitable?

### Prompt to ChatGPT

> Write pandas code to summarize BrightMart performance by `Region`. Include order count, total sales, total profit, average discount, and profit margin. Include missing region values if they exist.

In [11]:
region_summary = (
    df.groupby("Region", dropna=False)
      .agg(
          Orders=("OrderID", "count"),
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          AvgDiscount=("Discount", "mean")
      )
      .assign(ProfitMargin=lambda x: x["Profit"] / x["Sales"])
      .sort_values("Profit", ascending=False)
)

region_summary

,Orders,Sales,Profit,AvgDiscount,ProfitMargin
Region,,,,,
East,849,"2,491,995.07","201,298.65",0.14,0.08
West,874,"2,579,725.29","186,630.79",0.13,0.07
South,889,"2,376,625.83","170,982.05",0.13,0.07
Central,836,"2,406,377.69","165,697.65",0.13,0.07
west,65,"156,208.11","9,119.85",0.15,0.06
NaN,22,"52,753.84","4,359.99",0.11,0.08


### Instructor Note

This is a good place to discuss `dropna=False`. By default, pandas may exclude missing group labels. For a data quality discussion, missing regions should remain visible.

## Exercise 4 — Customer Segment Analysis

### Business Question

Which customer segments are most valuable?

### Prompt to ChatGPT

> I have a retail dataframe with `CustomerSegment`, `Sales`, `Profit`, `Quantity`, and `OrderID`.  
> Write pandas code to compare customer segments by total sales, total profit, order count, average order value, and average profit per order.

In [12]:
segment_summary = (
    df.groupby("CustomerSegment")
      .agg(
          Orders=("OrderID", "count"),
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Quantity=("Quantity", "sum")
      )
      .assign(
          AvgOrderValue=lambda x: x["Sales"] / x["Orders"],
          AvgProfitPerOrder=lambda x: x["Profit"] / x["Orders"],
          ProfitMargin=lambda x: x["Profit"] / x["Sales"]
      )
      .sort_values("Profit", ascending=False)
)

segment_summary

,Orders,Sales,Profit,Quantity,AvgOrderValue,AvgProfitPerOrder,ProfitMargin
CustomerSegment,,,,,,,
Home Office,1181,"3,340,953.66","260,986.86",7619,"2,828.92",220.99,0.08
Corporate,1160,"3,245,061.27","247,020.22",7837,"2,797.47",212.95,0.08
Consumer,1194,"3,477,670.90","230,081.90",7784,"2,912.62",192.70,0.07


### Reflection Question

If one segment has the highest sales but another has the highest profit margin, which should management prioritize? Why?

## Exercise 5 — Discount Impact

### Business Question

Do higher discounts appear to reduce profitability?

### Prompt to ChatGPT

> Using pandas, create discount bands from the `Discount` column and summarize order count, sales, profit, average profit margin, and negative-profit rate by discount band.

In [13]:
df["ProfitMargin"] = df["Profit"] / df["Sales"]

df["DiscountBand"] = pd.cut(
    df["Discount"],
    bins=[-0.001, 0.0, 0.10, 0.20, 1.00],
    labels=["No discount", "Low discount", "Medium discount", "High discount"]
)

discount_summary = (
    df.groupby("DiscountBand", observed=True)
      .agg(
          Orders=("OrderID", "count"),
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          AvgProfitMargin=("ProfitMargin", "mean"),
          NegativeProfitRate=("Profit", lambda s: (s < 0).mean())
      )
)

discount_summary

,Orders,Sales,Profit,AvgProfitMargin,NegativeProfitRate
DiscountBand,,,,,
No discount,581,"1,724,549.96","173,471.53",0.12,0.20
Low discount,1177,"3,203,470.17","309,357.02",0.11,0.24
Medium discount,1199,"3,484,904.34","221,162.07",0.07,0.33
High discount,578,"1,650,761.36","34,098.36",0.03,0.45


### Expected Output

A discount-band summary table. Students should explain whether higher discounts appear associated with lower margins or higher negative-profit rates.

**Important:** This is association, not proof of causation.

## Exercise 6 — Turn AI Code into Better Analyst Code

AI might generate code like this:

```python
df.groupby("Region").sum()
```

Ask students:
- What is wrong or incomplete?
- How can we make it clearer?
- Does it answer an executive business question?

In [14]:
# Improved analyst version
executive_region_summary = (
    df.groupby("Region", dropna=False)
      .agg(
          Orders=("OrderID", "count"),
          TotalSales=("Sales", "sum"),
          TotalProfit=("Profit", "sum"),
          AverageDiscount=("Discount", "mean")
      )
      .assign(ProfitMargin=lambda x: x["TotalProfit"] / x["TotalSales"])
      .sort_values("TotalProfit", ascending=False)
)

executive_region_summary

,Orders,TotalSales,TotalProfit,AverageDiscount,ProfitMargin
Region,,,,,
East,849,"2,491,995.07","201,298.65",0.14,0.08
West,874,"2,579,725.29","186,630.79",0.13,0.07
South,889,"2,376,625.83","170,982.05",0.13,0.07
Central,836,"2,406,377.69","165,697.65",0.13,0.07
west,65,"156,208.11","9,119.85",0.15,0.06
NaN,22,"52,753.84","4,359.99",0.11,0.08


## Exercise 7 — Business Interpretation

AI can write code, but analysts must explain what the numbers mean.

### Prompt to ChatGPT

> I have a pandas summary table showing Sales, Profit, and ProfitMargin by Region.  
> Give me a concise executive interpretation, but do not invent numbers. Tell me where I should insert the actual values from my table.

In [15]:
# Instructor example: generate interpretation from actual table values
top_region = executive_region_summary.index[0]
top_profit = executive_region_summary.iloc[0]["TotalProfit"]
top_margin = executive_region_summary.iloc[0]["ProfitMargin"]

print(
    f"The strongest region by total profit is {top_region}, "
    f"with total profit of ${top_profit:,.2f} and a profit margin of {top_margin:.2%}."
)

The strongest region by total profit is East, with total profit of $201,298.65 and a profit margin of 8.08%.


## Final Section 3 Checkpoint

Students should be able to complete these tasks without copying blindly from AI:

1. Monthly revenue table
2. Category summary
3. Region summary
4. Segment summary
5. Discount impact table
6. Verification checks
7. Business interpretation

They should also record at least two AI prompts in their prompt log.

## Common Student Mistakes

- Asking AI for code without listing column names.
- Forgetting to convert dates before grouping by month.
- Accepting `df.groupby(...).sum()` without selecting relevant columns.
- Not checking whether totals match after grouping.
- Treating correlation or association as causation.
- Reporting AI-written interpretations without checking the actual table values.

## Wrap-Up Discussion Questions

1. Which pandas task did AI help with most?
2. Which AI-generated code needed improvement?
3. What verification check was most useful?
4. How did adding business context improve the prompt?
5. What should students document in the prompt log?